In [9]:
!pip install tqdm

In [ ]:
import os
import shutil

root_path = "./datasets/manila"
output_path = os.path.join(root_path, "tiles")
os.makedirs(output_path, exist_ok=True)

# Loop through each folder inside root_path (no digit check)
for year in os.listdir(root_path):
    year_path = os.path.join(root_path, year)
    if not os.path.isdir(year_path):
        continue

    # Skip the "tiles" output folder to avoid recursion
    if year == "tiles":
        continue

    # Loop through quadrant folders
    for quadrant in ["1", "2", "3", "4"]:
        quad_path = os.path.join(year_path, quadrant)
        if not os.path.isdir(quad_path):
            continue

        # Loop through tile images
        for filename in os.listdir(quad_path):
            tile_id, ext = os.path.splitext(filename)
            if ext.lower() not in [ ".jpeg"]:
                continue

            # Make tile folder
            tile_folder = os.path.join(output_path, tile_id)
            os.makedirs(tile_folder, exist_ok=True)

            # Add year name to filename
            new_filename = f"{year}{ext}"
            new_filepath = os.path.join(tile_folder, new_filename)

            shutil.copy(os.path.join(quad_path, filename), new_filepath)

In [10]:
from tqdm import tqdm

# Collect all tiles first
all_tiles = []

for year in os.listdir(root_path):
    year_path = os.path.join(root_path, year)
    if not os.path.isdir(year_path) or year == "tiles":
        continue

    for quadrant in ["1", "2", "3", "4"]:
        quad_path = os.path.join(year_path, quadrant)
        if not os.path.isdir(quad_path):
            continue

        for filename in os.listdir(quad_path):
            tile_id, ext = os.path.splitext(filename)
            if ext.lower() in [".png", ".jpg", ".jpeg"]:
                all_tiles.append((year, quadrant, quad_path, filename))

# Move with progress bar
for year, quadrant, quad_path, filename in tqdm(all_tiles, desc="Moving tiles"):
    tile_id, ext = os.path.splitext(filename)

    tile_folder = os.path.join(output_path, tile_id)
    os.makedirs(tile_folder, exist_ok=True)

    new_filename = f"{year}{ext}"
    shutil.copy(
        os.path.join(quad_path, filename),
        os.path.join(tile_folder, new_filename)
    )


Moving tiles: 100%|██████████| 166059/166059 [28:09<00:00, 98.30it/s]  


In [5]:
from PIL import Image
import numpy as np


imf_path = "C:\\Users\\walaa\\Documents\\urban-evolution-ai\\ml-pipeline\\datasets\\manila\\ESRI 2014\\1\\out_219061_120239.jpeg"  # Example image path
def inspect_tile(img_path):
    img = Image.open(img_path).convert("RGB")
    arr = np.array(img, dtype=np.float32) / 255.0

    R = arr[:, :, 0]
    G = arr[:, :, 1]
    B = arr[:, :, 2]

    print("Mean R:", R.mean())
    print("Mean G:", G.mean())
    print("Mean B:", B.mean())

    print("Std  R:", R.std())
    print("Std  G:", G.std())
    print("Std  B:", B.std())

inspect_tile(imf_path)

Mean R: 0.03137255
Mean G: 0.14901961
Mean B: 0.19215685
Std  R: 0.0
Std  G: 0.0
Std  B: 1.4901161e-08


In [8]:
from PIL import Image
import numpy as np

def is_sea_tile(img_path, threshold=0.40):
    img = Image.open(img_path).convert("RGB")
    arr = np.array(img, dtype=np.float32) / 255.0

    R = arr[:, :, 0]
    G = arr[:, :, 1]
    B = arr[:, :, 2]

    # Calibré à partir de ta tile mer
    water_mask = (B > 0.16) & (R < 0.08) & (G < 0.22)

    water_ratio = water_mask.mean()
    return water_ratio > threshold

is_sea = is_sea_tile(imf_path)
print("Is sea tile:", is_sea)


Is sea tile: True


In [14]:
import os
import shutil
from pathlib import Path

tiles_root = Path("./datasets/manila/tiles")
Image.MAX_IMAGE_PIXELS = None 

deleted = 0
total = 0

for tile_dir in tiles_root.iterdir():
    if not tile_dir.is_dir():
        continue

    total += 1

    # Pick any image inside this tile folder to test
    candidate_img = None
    for fname in os.listdir(tile_dir):
        if fname.lower().endswith(".jpeg"):
            candidate_img = tile_dir / fname
            break

    if candidate_img is None:
        # No images? Just skip or delete as you like
        continue

    # Check if this tile is mostly sea
    if is_sea_tile(candidate_img):
        print(f"Deleting sea tile folder: {tile_dir}")
        shutil.rmtree(tile_dir)  # PERMANENT DELETE
        deleted += 1

print(f"Done. Deleted {deleted} / {total} tile folders.")

Done. Deleted 0 / 41852 tile folders.
